In [ ]:
!nvidia-smi
!pip -q install timm scikit-learn

 Load the data from Google Drive (low memory)

Images are copied to Colab's local disk and then read **one batch at a time** during training, instead of keeping
all of them in RAM.

In [ ]:
import os, gc, json, time, shutil, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DATA = "/content/drive/MyDrive/rfmid_256"
LOCAL_DATA = "/content/rfmid_256"
RESULTS = "/content/drive/MyDrive/rfmid_results"
os.makedirs(RESULTS, exist_ok=True)

if not os.path.exists(os.path.join(LOCAL_DATA, "dataset.csv")):
    t = time.time()
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print(f"Copied images to local disk in {time.time()-t:.0f} s")

ds = pd.read_csv(os.path.join(LOCAL_DATA, "dataset.csv"))
print(ds.shape)
print(ds.groupby("split")["Disease_Risk"].agg(["count", "sum"]).rename(columns={"sum": "abnormal"}))

## Settings and data pipeline

- **Augmentation** (training only): random crop, flips, small rotation, small brightness/contrast change.
- **Imbalance:** 79% abnormal, so the loss gives healthy photos a larger weight (`pos_weight`).
- The **same weighted loss** is used for training and validation, so the two loss curves are comparable.

In [ ]:
import torch, timm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} | {gpu_mem:.0f} GB")
else:
    gpu_name, gpu_mem = "cpu", 0
    print("WARNING: no GPU. Runtime > Change runtime type > pick a GPU, then run again.")

BATCH = 64 if gpu_mem >= 30 else 32     # A100 (40 GB) gets a bigger batch
IMG = 224
AMP = (device == "cuda")
print("Batch size:", BATCH)

MEANS = {"resnet50": (0.485, 0.456, 0.406), "vit": (0.5, 0.5, 0.5)}
STDS  = {"resnet50": (0.229, 0.224, 0.225), "vit": (0.5, 0.5, 0.5)}
TIMM_NAMES = {"resnet50": "resnet50.a1_in1k", "vit": "vit_base_patch16_224.augreg_in21k_ft_in1k"}

def make_tfms(kind):
    norm = T.Normalize(MEANS[kind], STDS[kind])
    train_tf = T.Compose([
        T.RandomResizedCrop(IMG, scale=(0.75, 1.0), ratio=(0.95, 1.05)),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(15),
        T.ColorJitter(brightness=0.15, contrast=0.15),
        T.ToTensor(), norm])
    eval_tf = T.Compose([T.Resize((IMG, IMG)), T.ToTensor(), norm])
    return train_tf, eval_tf

class RFMiD(Dataset):
    # reads each image from disk when needed (keeps RAM almost empty)
    def __init__(self, df, tf):
        self.paths = [os.path.join(LOCAL_DATA, p) for p in df["path"]]
        self.y = df["Disease_Risk"].values.astype("float32")
        self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            x = self.tf(im.convert("RGB"))
        return x, torch.tensor(self.y[i])

def make_loaders(kind):
    train_tf, eval_tf = make_tfms(kind)
    part = {s: ds[ds["split"] == s] for s in ["train", "val", "test"]}
    kw = dict(num_workers=2, pin_memory=(device == "cuda"), persistent_workers=False)
    return {
        "train": DataLoader(RFMiD(part["train"], train_tf), BATCH, shuffle=True, drop_last=True, **kw),
        "train_clean": DataLoader(RFMiD(part["train"], eval_tf), 64, shuffle=False, **kw),
        "val": DataLoader(RFMiD(part["val"], eval_tf), 64, shuffle=False, **kw),
        "test": DataLoader(RFMiD(part["test"], eval_tf), 64, shuffle=False, **kw)}

n_ab = int((ds[ds.split == "train"]["Disease_Risk"] == 1).sum())
n_he = int((ds[ds.split == "train"]["Disease_Risk"] == 0).sum())
POS_WEIGHT = n_he / n_ab
print(f"train abnormal={n_ab} healthy={n_he}  pos_weight={POS_WEIGHT:.3f}")

## Training and evaluation functions

In [ ]:
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix,
                             f1_score, balanced_accuracy_score, average_precision_score)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=device), reduction="sum")

def build_model(kind):
    return timm.create_model(TIMM_NAMES[kind], pretrained=True, num_classes=1).to(device)

@torch.no_grad()
def predict(model, loader):
    # returns mean loss, probabilities, labels
    model.eval(); probs, ys, tot = [], [], 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            logit = model(x).squeeze(1)
        logit = logit.float()
        tot += loss_fn(logit, y).item()
        probs.append(torch.sigmoid(logit).cpu().numpy()); ys.append(y.cpu().numpy())
    return tot / len(loader.dataset), np.concatenate(probs), np.concatenate(ys)

def best_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[np.argmax(tpr - fpr)])      # maximises sensitivity + specificity

def metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"auc": float(roc_auc_score(y, p)), "ap": float(average_precision_score(y, p)),
            "sensitivity": float(tp / (tp + fn)), "specificity": float(tn / (tn + fp)),
            "balanced_acc": float(balanced_accuracy_score(y, pred)), "f1": float(f1_score(y, pred)),
            "accuracy": float((pred == y).mean()), "threshold": float(thr),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

def train_one(kind, lr, wd, epochs, ckpt=None, loaders=None, verbose=True):
    # trains one model; keeps the epoch with the best validation AUC (saved to ckpt if given)
    set_seed()
    loaders = loaders or make_loaders(kind)
    model = build_model(kind)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    steps = epochs * len(loaders["train"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1, anneal_strategy="cos")
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    hist, best = [], {"val_auc": -1}
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); run = 0.0
        for x, y in loaders["train"]:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                logit = model(x).squeeze(1)
            loss = loss_fn(logit.float(), y)               # summed over the batch
            scaler.scale(loss / len(x)).backward()         # back-propagation (average loss per photo)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
        tr_aug = run / (len(loaders["train"]) * BATCH)
        tr_clean, _, _ = predict(model, loaders["train_clean"])
        va_loss, pv, yv = predict(model, loaders["val"])
        auc = float(roc_auc_score(yv, pv))
        row = {"epoch": ep, "train_loss_aug": tr_aug, "train_loss_clean": tr_clean,
               "val_loss": va_loss, "val_auc": auc}
        hist.append(row)
        flag = ""
        if auc > best["val_auc"]:
            best = dict(row)
            if ckpt: torch.save(model.state_dict(), ckpt)
            flag = "  <- best so far"
        if verbose:
            print(f"epoch {ep:2d}/{epochs} | train(aug) {tr_aug:.4f} | train(clean) {tr_clean:.4f} | "
                  f"val {va_loss:.4f} | val AUC {auc:.4f} | {time.time()-t0:.0f}s{flag}")
    del model, opt, sched, scaler
    gc.collect(); torch.cuda.empty_cache()
    return hist, best

def free_memory():
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
print("functions ready")

## Hyperparameter search for the ViT



In [ ]:
GRID = [
    {"lr": 1e-5, "wd": 0.05},
    {"lr": 3e-5, "wd": 0.05},
    {"lr": 1e-4, "wd": 0.05},
    {"lr": 3e-5, "wd": 0.01},
]
SEARCH_EPOCHS = 4
hp_file = os.path.join(RESULTS, "hp_search.csv")
done = pd.read_csv(hp_file).to_dict("records") if os.path.exists(hp_file) else []
loaders_vit = make_loaders("vit")

for cfg in GRID:
    if any(abs(d["lr"] - cfg["lr"]) < 1e-12 and abs(d["wd"] - cfg["wd"]) < 1e-12 for d in done):
        print("skip (already done):", cfg); continue
    print("\n--- trial:", cfg)
    hist, best = train_one("vit", cfg["lr"], cfg["wd"], SEARCH_EPOCHS, loaders=loaders_vit)
    done.append({**cfg, "best_val_auc": best["val_auc"], "val_loss_at_best": best["val_loss"],
                 "best_epoch": best["epoch"]})
    pd.DataFrame(done).to_csv(hp_file, index=False)

hp = pd.DataFrame(done).sort_values("best_val_auc", ascending=False).reset_index(drop=True)
display(hp)
BEST = {"lr": float(hp.loc[0, "lr"]), "wd": float(hp.loc[0, "wd"])}
print("Chosen hyperparameters:", BEST)

plt.figure(figsize=(7, 3.6))
labels = [f"lr={r.lr:g}\nwd={r.wd:g}" for r in hp.itertuples()]
plt.bar(labels, hp["best_val_auc"], color="#4c78a8")
plt.ylim(max(0.5, hp["best_val_auc"].min() - 0.05), 1.0)
plt.ylabel("best validation AUC"); plt.title("Hyperparameter search (ViT-B/16)")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, "hp_search.png"), dpi=150); plt.show()

## Train the final ViT-B/16 (best settings, 12 epochs)



In [ ]:
FINAL_EPOCHS = 12
hist_vit, best_vit = train_one("vit", BEST["lr"], BEST["wd"], FINAL_EPOCHS,
                               ckpt=os.path.join(RESULTS, "vit_best.pt"), loaders=loaders_vit)
pd.DataFrame(hist_vit).to_csv(os.path.join(RESULTS, "vit_history.csv"), index=False)
print("Best epoch:", best_vit)

##  ResNet-50 baseline



In [ ]:
RUN_BASELINE = False
del loaders_vit; free_memory()
if RUN_BASELINE:
    hist_res, best_res = train_one("resnet50", 1e-4, 0.05, 10, ckpt=os.path.join(RESULTS, "resnet50_best.pt"))
    pd.DataFrame(hist_res).to_csv(os.path.join(RESULTS, "resnet50_history.csv"), index=False)
    print("Best epoch:", best_res)
else:
    print("Baseline skipped (not needed for this assignment).")

##  Evaluate on the test set and save everything for the app



In [ ]:
def evaluate_saved(kind, tag):
    loaders = make_loaders(kind)
    model = build_model(kind)
    model.load_state_dict(torch.load(os.path.join(RESULTS, f"{tag}_best.pt"), map_location=device))
    vl, pv, yv = predict(model, loaders["val"])
    tl, pt, yt = predict(model, loaders["test"])
    thr = best_threshold(yv, pv)
    res = {"val": metrics(yv, pv, thr), "test": metrics(yt, pt, thr), "val_loss": vl, "test_loss": tl}
    pd.DataFrame({"ID": ds[ds.split == "test"]["ID"].values, "y": yt, "prob": pt}).to_csv(
        os.path.join(RESULTS, f"{tag}_test_preds.csv"), index=False)
    pd.DataFrame({"ID": ds[ds.split == "val"]["ID"].values, "y": yv, "prob": pv}).to_csv(
        os.path.join(RESULTS, f"{tag}_val_preds.csv"), index=False)
    json.dump(res, open(os.path.join(RESULTS, f"{tag}_results.json"), "w"), indent=1)
    del model, loaders; free_memory()
    return res

res_vit = evaluate_saved("vit", "vit")
res_res = evaluate_saved("resnet50", "resnet50") if RUN_BASELINE else None

# settings the Streamlit app needs
app_cfg = {"timm_name": TIMM_NAMES["vit"], "img_size": IMG, "mean": list(MEANS["vit"]), "std": list(STDS["vit"]),
           "threshold": res_vit["test"]["threshold"], "hyperparameters": BEST,
           "test_metrics": res_vit["test"], "seed": SEED}
json.dump(app_cfg, open(os.path.join(RESULTS, "app_config.json"), "w"), indent=1)

for name, r in [("ViT-B/16", res_vit)] + ([("ResNet-50", res_res)] if RUN_BASELINE else []):
    t = r["test"]
    print(f"\n=== {name}: TEST (threshold {t['threshold']:.3f}, chosen on validation) ===")
    for k in ["auc", "sensitivity", "specificity", "balanced_acc", "f1", "accuracy"]:
        print(f"  {k:13s} {t[k]:.3f}")
    print(f"  confusion: TN={t['tn']} FP={t['fp']} FN={t['fn']} TP={t['tp']}")

##  Loss curves (training vs validation)



In [ ]:
def loss_plot(hist, title, fname, ax=None):
    h = pd.DataFrame(hist)
    own = ax is None
    if own: fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(h["epoch"], h["train_loss_aug"], "o--", color="#9ecae9", label="train loss (augmented, running)")
    ax.plot(h["epoch"], h["train_loss_clean"], "o-", color="#1f77b4", label="train loss (clean)")
    ax.plot(h["epoch"], h["val_loss"], "s-", color="#d62728", label="validation loss")
    be = int(h.loc[h["val_auc"].idxmax(), "epoch"])
    ax.axvline(be, color="gray", ls=":", label=f"best epoch ({be}) by val AUC")
    ax.set_xlabel("epoch"); ax.set_ylabel("weighted BCE loss"); ax.set_title(title); ax.legend(fontsize=8)
    if own:
        plt.tight_layout(); plt.savefig(os.path.join(RESULTS, fname), dpi=150); plt.show()

loss_plot(hist_vit, "ViT-B/16: training vs validation loss", "vit_loss_curves.png")
if RUN_BASELINE:
    loss_plot(hist_res, "ResNet-50: training vs validation loss", "resnet50_loss_curves.png")

for name, hist in [("ViT-B/16", hist_vit)] + ([("ResNet-50", hist_res)] if RUN_BASELINE else []):
    h = pd.DataFrame(hist)
    above = int((h["val_loss"] > h["train_loss_clean"]).sum())
    print(f"{name}: validation loss is above the clean training loss in {above} of {len(h)} epochs "
          f"(final epoch: val {h['val_loss'].iloc[-1]:.4f} vs train {h['train_loss_clean'].iloc[-1]:.4f}).")
print()
print("How to read this: a validation loss that sits above the training loss is the normal sign that the model")
print("fits the training photos better than new photos (a generalisation gap). A growing gap means overfitting.")
print("If validation loss is NOT above training loss in the early epochs, that is also normal: the model is still")
print("under-trained, and the augmented training loss is made artificially hard. Report what you see.")

## Evaluation plots (test set)

In [ ]:
P = {"ViT-B/16": ("vit", res_vit, "#d62728")}
if RUN_BASELINE:
    P["ResNet-50"] = ("resnet50", res_res, "#1f77b4")
preds = {n: pd.read_csv(os.path.join(RESULTS, f"{tag}_test_preds.csv")) for n, (tag, _, _) in P.items()}

fig, ax = plt.subplots(2, 3, figsize=(18, 10))
# ROC
for n, (tag, r, c) in P.items():
    p = preds[n]; fpr, tpr, _ = roc_curve(p["y"], p["prob"])
    ax[0, 0].plot(fpr, tpr, color=c, label=f"{n} (AUC {r['test']['auc']:.3f})")
    t = r["test"]; ax[0, 0].plot(1 - t["specificity"], t["sensitivity"], "o", color=c)
ax[0, 0].plot([0, 1], [0, 1], "k--", alpha=.4); ax[0, 0].set_title("ROC curve (dots = chosen threshold)")
ax[0, 0].set_xlabel("1 - specificity"); ax[0, 0].set_ylabel("sensitivity"); ax[0, 0].legend()
# Precision-recall
for n, (tag, r, c) in P.items():
    p = preds[n]; pr, rc, _ = precision_recall_curve(p["y"], p["prob"])
    ax[0, 1].plot(rc, pr, color=c, label=f"{n} (AP {r['test']['ap']:.3f})")
ax[0, 1].set_title("Precision-recall curve"); ax[0, 1].set_xlabel("recall (sensitivity)"); ax[0, 1].set_ylabel("precision"); ax[0, 1].legend()
# Metric bars
mets = ["auc", "sensitivity", "specificity", "balanced_acc", "f1"]
w = 0.38
for i, (n, (tag, r, c)) in enumerate(P.items()):
    vals = [r["test"][m] for m in mets]
    b = ax[0, 2].bar(np.arange(len(mets)) + i * w, vals, w, color=c, label=n)
    for rect, v in zip(b, vals): ax[0, 2].text(rect.get_x() + w / 2, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
ax[0, 2].set_xticks(np.arange(len(mets)) + w / 2); ax[0, 2].set_xticklabels(["AUC", "sens.", "spec.", "bal. acc", "F1"])
ax[0, 2].set_ylim(0, 1.1); ax[0, 2].set_title("Test metrics"); ax[0, 2].legend()
# Confusion matrices (ViT and ResNet), probability histogram
for j, (n, (tag, r, c)) in enumerate(P.items()):
    t = r["test"]; cm = np.array([[t["tn"], t["fp"]], [t["fn"], t["tp"]]])
    a = ax[1, j]; a.imshow(cm, cmap="Blues")
    for i in range(2):
        for k in range(2):
            a.text(k, i, cm[i, k], ha="center", va="center", fontsize=18, color="white" if cm[i, k] > cm.max() / 2 else "black")
    a.set_xticks([0, 1]); a.set_xticklabels(["pred healthy", "pred abnormal"])
    a.set_yticks([0, 1]); a.set_yticklabels(["true healthy", "true abnormal"]); a.set_title(f"Confusion matrix: {n}")
for k in range(len(P), 2):
    ax[1, k].axis("off")
p = preds["ViT-B/16"]
ax[1, 2].hist(p[p.y == 0]["prob"], bins=20, alpha=.7, color="#2ca02c", label="truly healthy")
ax[1, 2].hist(p[p.y == 1]["prob"], bins=20, alpha=.7, color="#d62728", label="truly abnormal")
ax[1, 2].axvline(res_vit["test"]["threshold"], color="k", ls="--", label="threshold")
ax[1, 2].set_title("ViT: predicted probability of 'abnormal'"); ax[1, 2].set_xlabel("probability"); ax[1, 2].legend()
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, "evaluation_dashboard.png"), dpi=150); plt.show()

# summary table
rows = [{"model": n, **{m: round(r["test"][m], 3) for m in ["auc", "sensitivity", "specificity", "balanced_acc", "f1", "accuracy"]}}
        for n, (tag, r, c) in P.items()]
table = pd.DataFrame(rows); table.to_csv(os.path.join(RESULTS, "test_metrics.csv"), index=False)
display(table)

In [ ]:
# Per-disease sensitivity of the ViT on the test set (how many photos with each disease are flagged abnormal)
test_df = ds[ds.split == "test"].reset_index(drop=True)
pt = preds["ViT-B/16"]
flag = (pt["prob"].values >= res_vit["test"]["threshold"]).astype(int)
non_disease = {"ID", "Disease_Risk", "split", "path", "orig_w", "orig_h", "brightness", "contrast"}
rows = []
for c in [c for c in test_df.columns if c not in non_disease]:
    m = test_df[c].values == 1
    if m.sum() >= 8:
        rows.append({"disease": c, "n_test_photos": int(m.sum()), "flagged_abnormal": float(flag[m].mean())})
pdz = pd.DataFrame(rows).sort_values("flagged_abnormal")
pdz.to_csv(os.path.join(RESULTS, "per_disease_sensitivity.csv"), index=False)
plt.figure(figsize=(8, max(3, 0.35 * len(pdz) + 1)))
plt.barh([f"{d} (n={n})" for d, n in zip(pdz["disease"], pdz["n_test_photos"])], pdz["flagged_abnormal"], color="#4c78a8")
plt.xlim(0, 1.05); plt.xlabel("share flagged as abnormal"); plt.title("ViT: detection rate per disease (test set)")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, "per_disease_sensitivity.png"), dpi=150); plt.show()
display(pdz)